# Experiment 2 — Location State Extraction (Single Model)

Evaluates **one PaliGemma model** on the UFV Campus dataset for the task of extracting location information from indoor signage — building codes, floor numbers, and room codes.

---

## What this notebook does

Two categories of images are evaluated with 4 questions total:

| Category | Images | Questions | Task |
|----------|--------|-----------|------|
| `building_sign` | 12 | 1 (combined) | Extract building code + floor number from sign |
| `door_room` | 10 | 3 | Detect room code, read it, detect door state |

**Image types in `building_sign`:**

| Type | Images | Description |
|------|--------|-------------|
| `complete` | 01–06 | Full sign with building code (EPS/G) + floor |
| `sign_only` | 07–09 | Floor number on sign, no building code |
| `wall_number` | 10–12 | Large floor number painted on wall |

**Metrics computed:**

- **Building Accuracy** — building code correctly extracted (flexible match via `building_acceptable`)
- **Floor Accuracy** — floor number correctly extracted from response
- **Location Accuracy** — building AND floor both correct (type=complete only)
- **BERTScore F1/P/R** — semantic similarity of full response vs reference (type=complete only)
- **Room Code Accuracy** — expected room code found in prediction (via `room_code_acceptable`)
- **Exact Match** — for `doorroom_detection` (yes/no) and `doorroom_state` (open/closed)
- **Floor confusion matrix** — predicted vs expected floor (−2,−1,0,1,2,other)
- **Door state matrix** — open/closed confusion for `doorroom_state`
- **Type-based accuracy breakdown** — complete vs sign_only vs wall_number
- **None-hallucination rate** — does model invent a building code when none is visible?
- **Efficiency** — VRAM peak, avg inference time, throughput

---

## How to run

1. Open in **Google Colab** (GPU runtime — T4 for 3B models, A100 for 10B)
2. Upload `images.zip` and `annotations.zip` to `/content/`
3. Run **Cell 2** (Unzip) to extract both archives
4. Set `HF_TOKEN` and `MODEL_ID` in **Cell 3** (Configuration) — only change needed
5. Run all cells top to bottom
6. Download `/content/results/exp2/` as ZIP when complete

**To evaluate a different model**, only change `MODEL_ID` in Cell 3.

---

## Output files (per model)

```
/content/results/exp2/
├── {model}_exp2_results.csv        ← per-image, per-question predictions + metrics
├── {model}_exp2_aggregate.csv      ← metrics aggregated by category
├── {model}_exp2_efficiency.json    ← VRAM, timing, throughput
├── {model}_exp2_matrices.json      ← floor matrix, door state matrix, type breakdown
└── plots/
    ├── {model}_accuracy_by_category.png
    ├── {model}_floor_confusion.png
    ├── {model}_door_state_confusion.png
    ├── {model}_type_breakdown.png
    └── {model}_room_code_accuracy.png
```

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────
!pip install -q transformers torch Pillow bert-score accelerate

In [ ]:
# ── 2. Unzip datasets ──────────────────────────────────────────
# Upload images.zip and annotations.zip to /content/ before running
import zipfile, os

def unzip_if_needed(zip_path, extract_to):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'Already extracted: {extract_to}')
        return
    print(f'Extracting {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    print(f'Done: {extract_to}')

unzip_if_needed('/content/images.zip',      '/content/images')
unzip_if_needed('/content/annotations.zip', '/content/annotations')

print('\nImages subfolders:',      [f for f in os.listdir('/content/images')      if os.path.isdir(f'/content/images/{f}')])
print('Annotation files:',         os.listdir('/content/annotations'))

In [ ]:
# ── 3. Configuration — only change MODEL_ID to switch models ───
import os

MODEL_ID = 'google/paligemma2-3b-mix-448'
HF_TOKEN  = ''   # Set your HuggingFace token here

IMAGES_BASE      = '/content/images'
ANNOTATIONS_BASE = '/content/annotations'
RESULTS_PATH     = '/content/results/exp2'

MAX_NEW_TOKENS = 30


def _resolve_ann_path(base):
    # If zip contained a folder 'annotations/', JSON files are in base/annotations/
    if os.path.isfile(os.path.join(base, 'questions.json')):
        return base
    nested = os.path.join(base, 'annotations')
    if os.path.isfile(os.path.join(nested, 'questions.json')):
        print(f'Note: annotations inside subfolder, using {nested}')
        return nested
    return base


def _resolve_images_path(base):
    # If zip contained a folder 'images/', categories are in base/images/
    def has_exp2(root):
        return os.path.isdir(os.path.join(root, 'building_sign')) or os.path.isdir(
            os.path.join(root, 'door_room')
        )

    if has_exp2(base):
        return base
    nested = os.path.join(base, 'images')
    if has_exp2(nested):
        print(f'Note: image categories inside subfolder, using {nested}')
        return nested
    return base


IMAGES_PATH      = _resolve_images_path(IMAGES_BASE)
ANNOTATIONS_PATH = _resolve_ann_path(ANNOTATIONS_BASE)

print(f'Model:       {MODEL_ID}')
print(f'Images:      {IMAGES_PATH}')
print(f'Annotations: {ANNOTATIONS_PATH}')
print(f'Results:     {RESULTS_PATH}')
print(f'Max tokens:  {MAX_NEW_TOKENS}')


In [ ]:
# ── 4. Imports ─────────────────────────────────────────────────
try:
    from IPython.display import display
except ImportError:
    display = print

import os, json, time, re
import torch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor
from bert_score import score as bert_score_fn

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)
print('Imports OK')

## Model Loading

In [ ]:
# ── 5. Load model and processor ────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

load_start = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32
).to(device).eval()
model_load_time_s = round(time.time() - load_start, 2)

n_parameters = sum(p.numel() for p in model.parameters())
vram_load_gb = torch.cuda.max_memory_allocated() / (1024**3) if device == 'cuda' else 0
print(f'Model loaded: {MODEL_ID}')
print(f'Parameters: {n_parameters:,} ({n_parameters/1e9:.2f}B)')
print(f'VRAM after load: {vram_load_gb:.2f} GB')
print(f'Load time: {model_load_time_s:.1f}s')

## Data & Annotations

In [ ]:
# ── 6. Load questions and expected answers ─────────────────────
with open(f'{ANNOTATIONS_PATH}/questions.json') as f:
    questions_data = json.load(f)

with open(f'{ANNOTATIONS_PATH}/exp2_expected_answers.json') as f:
    expected_answers = json.load(f)

building_sign_q = questions_data['experiment2']['categories']['building_sign']['questions'][0]
door_room_qs   = questions_data['experiment2']['categories']['door_room']['questions']

building_sign_answers = expected_answers['building_sign']
door_room_answers     = expected_answers['door_room']

print('Questions loaded')
print(f'  building_sign: 1 question — "{building_sign_q["id"]}"')
print(f'  door_room:     {len(door_room_qs)} questions — {[q["id"] for q in door_room_qs]}')
print(f'Building sign images: {len(building_sign_answers)}')
print(f'Door room images:     {len(door_room_answers)}')

## Core Functions

In [ ]:
# ── 7. Inference function ──────────────────────────────────────
def run_inference(image, question):
    prompt = f'<image>{question}'
    inputs = processor(
        text=prompt, images=image,
        padding='longest', do_convert_rgb=True,
        return_tensors='pt'
    ).to(device)
    inputs = inputs.to(dtype=model.dtype)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    input_len = inputs['input_ids'].shape[-1]
    return processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

print('Inference function defined')

In [ ]:
# ── 8. Metric functions ────────────────────────────────────────
VALID_FLOORS = {'-2', '-1', '0', '1', '2'}

def normalize(text):
    return str(text).lower().strip()

def extract_floor_from_response(prediction):
    '''Extract valid floor number from response. Handles: EPS 0, -1, G 1, eps0, 2.'''    
    pred = normalize(prediction)
    # Token match first (most common: model outputs '0', '-1', 'EPS 0')
    for token in pred.split():
        if token in VALID_FLOORS:
            return token
    # Regex fallback for concatenated responses like 'eps0', 'g1'
    for m in re.finditer(r'-?\d+', pred):
        if m.group() in VALID_FLOORS:
            return m.group()
    return None

def calc_building_accuracy(prediction, expected_building, building_acceptable):
    '''1.0 if any acceptable building variant found in prediction. None if no building expected.'''
    if expected_building is None:
        return None
    pred = normalize(prediction)
    for variant in (building_acceptable or []):
        if normalize(str(variant)) in pred:
            return 1.0
    return 0.0

def calc_floor_accuracy(prediction, expected_floor):
    '''1.0 if extracted floor matches expected floor. None if no floor expected.'''
    if expected_floor is None:
        return None
    extracted = extract_floor_from_response(prediction)
    return 1.0 if extracted == str(expected_floor) else 0.0

def calc_location_accuracy(building_ok, floor_ok):
    '''1.0 if both building and floor are correct. None if either is not applicable.'''
    if building_ok is None or floor_ok is None:
        return None
    return 1.0 if (building_ok == 1.0 and floor_ok == 1.0) else 0.0

def calc_room_code_accuracy(prediction, expected_code, room_code_acceptable):
    '''1.0 if expected room code or any acceptable variant found in prediction.'''
    pred = normalize(prediction)
    for variant in (room_code_acceptable or []):
        if normalize(str(variant)) in pred:
            return 1.0
    if normalize(str(expected_code)) in pred:
        return 1.0
    return 0.0

def calc_exact_match(prediction, expected):
    """Robust exact match for closed-vocabulary and yes/no questions.

    Uses word-boundary matching to avoid substring leaks (so 'up' does not
    match 'upper'), and prioritises negation cues so responses like
    'there is no door' are not counted as positive.
    """
    import re
    pred = normalize(prediction)
    exp = normalize(expected)
    if not pred or not exp:
        return 0.0

    if exp in ('yes', 'no'):
        is_neg = bool(re.search(
            r"\bno\b|\bnot\b|\bnone\b|\bnever\b|\bcannot\b|\bcan't\b"
            r"|\bthere (?:is|are) no\b"
            r"|\bi (?:do not|don't|cannot|can't) see\b"
            r"|\bnot (?:visible|present)\b",
            pred,
        ))
        is_pos = bool(re.search(
            r"\byes\b|\byeah\b|\byep\b"
            r"|\bi (?:can see|see)\b"
            r"|\bthere (?:is|are) (?:a |an |some )?\w"
            r"|\b(?:visible|present)\b"
            r"|\bit is\b|\bthis is\b|\bthat is\b",
            pred,
        )) and not is_neg
        if exp == 'yes':
            return 1.0 if is_pos else 0.0
        return 1.0 if (is_neg and not is_pos) else 0.0

    tokens = pred.split()
    if tokens and tokens[0] == exp:
        return 1.0
    return 1.0 if re.search(r"\b" + re.escape(exp) + r"\b", pred) else 0.0


def is_building_hallucinated(prediction):
    """True if the prediction contains a UFV building code as a whole word.

    Strips punctuation by tokenising on alphabetic word boundaries, so
    responses like 'Building G.' or 'G,' are detected correctly.
    """
    import re
    pred = normalize(prediction)
    tokens = re.findall(r"\b[a-z]+\b", pred)
    codes = {'eps', 'g', 'cd', 'h'}
    return any(t in codes for t in tokens)


print('Metric functions defined')

## Inference Loop

In [ ]:
# ── 9. Run Experiment 2 — inference loop ───────────────────────
results = []
total_start = time.time()

# ── building_sign (1 question per image) ──────────────────────
print('Category: building_sign')
bs_question = building_sign_q['question']

for img_name, img_data in building_sign_answers.items():
    img_path = os.path.join(IMAGES_PATH, 'building_sign', img_name)
    if not os.path.exists(img_path):
        print(f'  WARNING: not found: {img_path}')
        continue

    image          = Image.open(img_path).convert('RGB')
    exp_building   = img_data.get('building')
    exp_floor      = img_data.get('floor')
    acceptable_b   = img_data.get('building_acceptable', [])
    bertscore_ref  = img_data.get('bertscore_reference')   # None for sign_only/wall_number
    img_type       = img_data.get('type', 'unknown')

    start = time.time()
    prediction = run_inference(image, bs_question)
    inf_ms = (time.time() - start) * 1000

    b_ok  = calc_building_accuracy(prediction, exp_building, acceptable_b)
    f_ok  = calc_floor_accuracy(prediction, exp_floor)
    loc_ok = calc_location_accuracy(b_ok, f_ok)
    hall  = is_building_hallucinated(prediction) if exp_building is None else None

    results.append({
        'model_id': MODEL_ID,
        'category': 'building_sign',
        'image': img_name,
        'image_type': img_type,
        'question_id': 'building_sign_combined',
        'question': bs_question,
        'prediction': prediction,
        'expected_building': exp_building,
        'expected_floor': exp_floor,
        'expected_value': bertscore_ref or (f"{exp_building} {exp_floor}" if exp_building else exp_floor),
        'inference_time_ms': round(inf_ms, 2),
        'output_length_words': len(prediction.split()),
        'building_accuracy': b_ok,
        'floor_accuracy': f_ok,
        'location_accuracy': loc_ok,
        'hallucination': hall,
        'room_code_accuracy': None,
        'exact_match': None,
        'bertscore_f1': None,
        'bertscore_p': None,
        'bertscore_r': None,
        'bertscore_reference': bertscore_ref
    })
    label = 'halluc' if hall else ('ok' if f_ok == 1.0 else 'fail')
    print(f'  {img_name} [{img_type}] -> "{prediction}" [{label}]')

# ── door_room (3 questions per image) ─────────────────────────
print('\nCategory: door_room')

for img_name, img_data in door_room_answers.items():
    img_path = os.path.join(IMAGES_PATH, 'door_room', img_name)
    if not os.path.exists(img_path):
        print(f'  WARNING: not found: {img_path}')
        continue

    image = Image.open(img_path).convert('RGB')

    for q_data in door_room_qs:
        q_id      = q_data['id']
        question  = q_data['question']

        start = time.time()
        prediction = run_inference(image, question)
        inf_ms = (time.time() - start) * 1000

        row = {
            'model_id': MODEL_ID,
            'category': 'door_room',
            'image': img_name,
            'image_type': 'door_room',
            'question_id': q_id,
            'question': question,
            'prediction': prediction,
            'expected_building': None,
            'expected_floor': None,
            'inference_time_ms': round(inf_ms, 2),
            'output_length_words': len(prediction.split()),
            'building_accuracy': None,
            'floor_accuracy': None,
            'location_accuracy': None,
            'hallucination': None,
            'bertscore_f1': None,
            'bertscore_p': None,
            'bertscore_r': None,
            'bertscore_reference': None
        }

        if q_id == 'doorroom_detection':
            exp = img_data.get('doorroom_detection', 'yes')
            row.update({'expected_value': exp, 'exact_match': calc_exact_match(prediction, exp), 'room_code_accuracy': None})
        elif q_id == 'doorroom_code':
            exp_code   = img_data.get('room_code')
            acceptable = img_data.get('room_code_acceptable', [])
            row.update({'expected_value': exp_code, 'room_code_accuracy': calc_room_code_accuracy(prediction, exp_code, acceptable), 'exact_match': None})
        elif q_id == 'doorroom_state':
            exp = img_data.get('doorroom_state')
            row.update({'expected_value': exp, 'exact_match': calc_exact_match(prediction, exp), 'room_code_accuracy': None})

        results.append(row)

total_time_s = time.time() - total_start
df = pd.DataFrame(results)
print(f'\nTotal inferences: {len(df)} in {total_time_s:.1f}s')
print(f'Images processed: {df["image"].nunique()}')

## BERTScore

In [ ]:
# ── 10. BERTScore — building_sign complete images only ─────────
bs_mask = df['bertscore_reference'].notna() & (df['bertscore_reference'] != '')
n_bs = bs_mask.sum()

if n_bs > 0:
    print(f'Computing BERTScore for {n_bs} predictions...')
    P, R, F1 = bert_score_fn(
        df[bs_mask]['prediction'].tolist(),
        df[bs_mask]['bertscore_reference'].tolist(),
        lang='en', verbose=False
    )
    df.loc[bs_mask, 'bertscore_f1'] = F1.numpy()
    df.loc[bs_mask, 'bertscore_p']  = P.numpy()
    df.loc[bs_mask, 'bertscore_r']  = R.numpy()
    print(f'BERTScore mean F1: {F1.mean():.4f}')
else:
    print('No BERTScore references found (only complete-type images have references)')

## Aggregate Metrics

In [ ]:
# ── 11. Aggregate metrics per category ─────────────────────────
print('AGGREGATE METRICS')
print('=' * 60)

agg_rows = []

# ── building_sign ─────────────────────────────────────────────
bs_df = df[df['category'] == 'building_sign']
print('\nBUILDING_SIGN')

agg_bs = {'category': 'building_sign'}

b_vals = bs_df['building_accuracy'].dropna()
if len(b_vals):
    agg_bs['building_accuracy'] = round(b_vals.mean(), 4)
    agg_bs['building_n']        = len(b_vals)
    print(f'  Building Accuracy: {agg_bs["building_accuracy"]:.3f} (n={len(b_vals)})')

f_vals = bs_df['floor_accuracy'].dropna()
if len(f_vals):
    agg_bs['floor_accuracy'] = round(f_vals.mean(), 4)
    agg_bs['floor_n']        = len(f_vals)
    print(f'  Floor Accuracy:    {agg_bs["floor_accuracy"]:.3f} (n={len(f_vals)})')

loc_vals = bs_df['location_accuracy'].dropna()
if len(loc_vals):
    agg_bs['location_accuracy'] = round(loc_vals.mean(), 4)
    agg_bs['location_n']        = len(loc_vals)
    print(f'  Location Accuracy: {agg_bs["location_accuracy"]:.3f} (n={len(loc_vals)})')

bs_vals = bs_df['bertscore_f1'].dropna()
if len(bs_vals):
    agg_bs['bertscore_f1'] = round(bs_vals.mean(), 4)
    agg_bs['bertscore_p']  = round(bs_df['bertscore_p'].dropna().mean(), 4)
    agg_bs['bertscore_r']  = round(bs_df['bertscore_r'].dropna().mean(), 4)
    print(f'  BERTScore F1:      {agg_bs["bertscore_f1"]:.3f} (n={len(bs_vals)})')

# Accuracy by image type
for img_type in ['complete', 'sign_only', 'wall_number']:
    sub = bs_df[bs_df['image_type'] == img_type]
    if len(sub) == 0:
        continue
    f_acc = sub['floor_accuracy'].dropna().mean()
    agg_bs[f'{img_type}_floor_accuracy'] = round(f_acc, 4) if not pd.isna(f_acc) else None
    if img_type == 'complete':
        b_acc = sub['building_accuracy'].dropna().mean()
        agg_bs[f'{img_type}_building_accuracy'] = round(b_acc, 4) if not pd.isna(b_acc) else None
        loc_acc = sub['location_accuracy'].dropna().mean()
        agg_bs[f'{img_type}_location_accuracy'] = round(loc_acc, 4) if not pd.isna(loc_acc) else None
    print(f'  [{img_type}] floor={f_acc:.3f} (n={len(sub)})')

# Hallucination rate (sign_only + wall_number only)
hall_df = bs_df[bs_df['hallucination'].notna()]
if len(hall_df):
    hall_rate = hall_df['hallucination'].mean()
    agg_bs['hallucination_rate'] = round(hall_rate, 4)
    print(f'  Hallucination rate (no-building images): {hall_rate:.3f} ({hall_df["hallucination"].sum():.0f}/{len(hall_df)})')

agg_bs['avg_output_length'] = round(bs_df['output_length_words'].mean(), 2)
agg_rows.append(agg_bs)

# ── door_room ─────────────────────────────────────────────────
dr_df = df[df['category'] == 'door_room']
print('\nDOOR_ROOM')

agg_dr = {'category': 'door_room'}

for q_id, label in [('doorroom_detection', 'Detection EM'), ('doorroom_state', 'State EM')]:
    sub = dr_df[dr_df['question_id'] == q_id]
    em_vals = sub['exact_match'].dropna()
    if len(em_vals):
        key = q_id + '_accuracy'
        agg_dr[key] = round(em_vals.mean(), 4)
        print(f'  {label}: {agg_dr[key]:.3f} (n={len(em_vals)})')

rc_sub  = dr_df[dr_df['question_id'] == 'doorroom_code']
rc_vals = rc_sub['room_code_accuracy'].dropna()
if len(rc_vals):
    agg_dr['room_code_accuracy'] = round(rc_vals.mean(), 4)
    print(f'  Room Code Accuracy: {agg_dr["room_code_accuracy"]:.3f} (n={len(rc_vals)})')

agg_dr['avg_output_length'] = round(dr_df['output_length_words'].mean(), 2)
agg_rows.append(agg_dr)

agg_df = pd.DataFrame(agg_rows)
print('\nSUMMARY TABLE')
display(agg_df)

## Per-Image Results

In [ ]:
# ── 12. Per-image detailed results ─────────────────────────────
print('PER-IMAGE RESULTS')
print('=' * 60)
cols = ['category', 'image', 'image_type', 'question_id',
        'prediction', 'expected_value',
        'building_accuracy', 'floor_accuracy', 'location_accuracy',
        'room_code_accuracy', 'exact_match', 'bertscore_f1', 'hallucination']
display(df[[c for c in cols if c in df.columns]])

## Efficiency Metrics

In [ ]:
# ── 13. Efficiency metrics ─────────────────────────────────────
vram_peak_gb = torch.cuda.max_memory_allocated() / (1024**3) if device == 'cuda' else 0
n_images     = df['image'].nunique()
total_inf_ms = df['inference_time_ms'].sum()

efficiency = {
    'model_id': MODEL_ID, 'experiment': 'exp2',
    'n_parameters': n_parameters,
    'n_parameters_billions': round(n_parameters / 1e9, 2),
    'n_images': n_images, 'n_inferences': len(df),
    'vram_peak_gb': round(vram_peak_gb, 3),
    'model_load_time_s': model_load_time_s,
    'total_wall_time_s': round(total_time_s, 2),
    'total_inference_time_s': round(total_inf_ms / 1000, 2),
    'avg_inference_time_ms': round(df['inference_time_ms'].mean(), 2),
    'throughput_img_per_s': round(n_images / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    'throughput_inf_per_s': round(len(df) / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    'avg_output_length_words': round(df['output_length_words'].mean(), 2),
    'max_output_length_words': int(df['output_length_words'].max())
}

print('EFFICIENCY METRICS')
print('=' * 40)
for k, v in efficiency.items():
    print(f'  {k}: {v}')

## Save Results

In [ ]:
# ── 14. Save CSV and JSON results ──────────────────────────────
os.makedirs(RESULTS_PATH, exist_ok=True)
model_name_safe = MODEL_ID.replace('/', '_').replace('-', '_')

results_file    = f'{RESULTS_PATH}/{model_name_safe}_exp2_results.csv'
agg_file        = f'{RESULTS_PATH}/{model_name_safe}_exp2_aggregate.csv'
efficiency_file = f'{RESULTS_PATH}/{model_name_safe}_exp2_efficiency.json'

df.to_csv(results_file, index=False)
agg_df.to_csv(agg_file, index=False)
with open(efficiency_file, 'w') as f:
    json.dump(efficiency, f, indent=2)

print(f'Saved: {results_file}')
print(f'Saved: {agg_file}')
print(f'Saved: {efficiency_file}')

## Confusion Matrices & Analysis

In [ ]:
# ── 15. Confusion matrices ─────────────────────────────────────

# ── Floor confusion matrix ─────────────────────────────────────
floor_labels = ['-2', '-1', '0', '1', '2', 'other']
fl_idx = {l: i for i, l in enumerate(floor_labels)}
n = len(floor_labels)
floor_matrix = [[0]*n for _ in range(n)]
f_correct, f_total = 0, 0

for _, row in df[df['category'] == 'building_sign'].iterrows():
    exp_f = str(row['expected_floor']) if row['expected_floor'] is not None else None
    if exp_f is None:
        continue
    pred_f = extract_floor_from_response(str(row['prediction']))
    pred_label = pred_f if pred_f in fl_idx else 'other'
    ei = fl_idx.get(exp_f)
    pi = fl_idx.get(pred_label)
    if ei is None:
        continue
    f_total += 1
    floor_matrix[ei][pi] += 1
    if ei == pi:
        f_correct += 1

floor_acc = f_correct / f_total if f_total > 0 else 0.0
print('FLOOR CONFUSION MATRIX (rows=expected, cols=predicted)')
print(f'Accuracy: {floor_acc:.3f} ({f_correct}/{f_total})')
print(f"{'':>8}" + ''.join(f'{l:>8}' for l in floor_labels))
for i, label in enumerate(floor_labels):
    print(f'{label:>8}' + ''.join(f'{floor_matrix[i][j]:>8}' for j in range(n)))

# ── Door state confusion matrix ────────────────────────────────
state_labels = ['open', 'closed']
st_idx = {'open': 0, 'closed': 1}
state_matrix = [[0, 0], [0, 0]]
s_correct, s_total, s_unknown = 0, 0, 0

for _, row in df[df['question_id'] == 'doorroom_state'].iterrows():
    exp = normalize(str(row['expected_value']))
    tokens = normalize(str(row['prediction'])).split()
    pred = tokens[0] if tokens else ''
    ei = st_idx.get(exp)
    pi = st_idx.get(pred)
    if ei is None:
        continue
    s_total += 1
    if pi is not None:
        state_matrix[ei][pi] += 1
        if ei == pi:
            s_correct += 1
    else:
        s_unknown += 1

state_acc = s_correct / s_total if s_total > 0 else 0.0
print('\nDOOR STATE CONFUSION MATRIX (rows=expected, cols=predicted)')
print(f'Accuracy: {state_acc:.3f} ({s_correct}/{s_total})  Unknown: {s_unknown}')
print(f"{'':>10}{'open':>10}{'closed':>10}")
for i, label in enumerate(state_labels):
    print(f'{label:>10}' + ''.join(f'{state_matrix[i][j]:>10}' for j in range(2)))

In [ ]:
# ── 16. Type breakdown + hallucination analysis ─────────────────
print('ACCURACY BY IMAGE TYPE (building_sign)')
print('=' * 50)
for img_type in ['complete', 'sign_only', 'wall_number']:
    sub = df[(df['category'] == 'building_sign') & (df['image_type'] == img_type)]
    if len(sub) == 0:
        continue
    f_acc = sub['floor_accuracy'].dropna().mean()
    print(f'  [{img_type}] n={len(sub)}')
    if img_type == 'complete':
        b_acc   = sub['building_accuracy'].dropna().mean()
        loc_acc = sub['location_accuracy'].dropna().mean()
        bs_acc  = sub['bertscore_f1'].dropna().mean()
        print(f'    Building Accuracy:  {b_acc:.3f}')
        print(f'    Floor Accuracy:     {f_acc:.3f}')
        print(f'    Location Accuracy:  {loc_acc:.3f}')
        print(f'    BERTScore F1:       {bs_acc:.3f}')
    else:
        print(f'    Floor Accuracy: {f_acc:.3f}')

print('\nNONE-HALLUCINATION ANALYSIS')
print('(Images where no building code is visible — model should NOT output EPS/G/CD/H)')
hall_df = df[(df['category'] == 'building_sign') & df['hallucination'].notna()]
if len(hall_df):
    for _, r in hall_df.iterrows():
        status = 'HALLUCINATED' if r['hallucination'] else 'OK'
        print(f'  {r["image"]} [{r["image_type"]}]: "{r["prediction"]}" [{status}]')
    rate = hall_df['hallucination'].mean()
    print(f'\n  Hallucination rate: {rate:.3f} ({int(hall_df["hallucination"].sum())}/{len(hall_df)})')
else:
    print('  No non-building images in dataset')

print('\nROOM CODE ACCURACY BREAKDOWN')
rc_df = df[df['question_id'] == 'doorroom_code']
for _, r in rc_df.iterrows():
    status = 'OK' if r['room_code_accuracy'] == 1.0 else 'FAIL'
    print(f'  {r["image"]}: expected="{r["expected_value"]}" pred="{r["prediction"]}" [{status}]')

In [ ]:
# ── 17. Save confusion matrices to JSON ────────────────────────
matrices = {
    'model_id': MODEL_ID, 'experiment': 'exp2',
    'floor_confusion': {
        'labels': floor_labels,
        'matrix': floor_matrix,
        'accuracy': round(floor_acc, 4),
        'correct': f_correct, 'total': f_total,
        'note': 'Rows = expected floor, columns = predicted floor. other = extracted value not in valid set.'
    },
    'door_state': {
        'labels': state_labels,
        'matrix': state_matrix,
        'accuracy': round(state_acc, 4),
        'correct': s_correct, 'total': s_total, 'unknown': s_unknown,
        'note': 'Rows = expected state, columns = predicted state (first token).'
    },
    'type_breakdown': {
        img_type: {
            'n': int(len(df[(df['category']=='building_sign') & (df['image_type']==img_type)])),
            'floor_accuracy': round(df[(df['category']=='building_sign') & (df['image_type']==img_type)]['floor_accuracy'].dropna().mean(), 4)
            if len(df[(df['category']=='building_sign') & (df['image_type']==img_type)]['floor_accuracy'].dropna()) > 0 else None
        }
        for img_type in ['complete', 'sign_only', 'wall_number']
    },
    'hallucination': {
        'rate': round(hall_df['hallucination'].mean(), 4) if len(hall_df) > 0 else None,
        'n_hallucinated': int(hall_df['hallucination'].sum()) if len(hall_df) > 0 else 0,
        'n_total': len(hall_df)
    }
}

matrices_file = f'{RESULTS_PATH}/{model_name_safe}_exp2_matrices.json'
with open(matrices_file, 'w') as f:
    json.dump(matrices, f, indent=2)
print(f'Saved: {matrices_file}')

## Visualizations

In [ ]:
# ── 18. Generate plots ─────────────────────────────────────────
plots_path = f'{RESULTS_PATH}/plots'
os.makedirs(plots_path, exist_ok=True)

def save_heatmap(matrix, labels_row, labels_col, title, filepath, cmap='Blues', annot_fs=13):
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=(max(5, len(labels_col)*1.4), max(4, len(labels_row)*1.2)))
    im = ax.imshow(arr, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels_col))); ax.set_yticks(range(len(labels_row)))
    ax.set_xticklabels(labels_col, fontsize=10); ax.set_yticklabels(labels_row, fontsize=10)
    ax.set_xlabel('Predicted', fontsize=11); ax.set_ylabel('Ground Truth', fontsize=11)
    ax.set_title(title, fontsize=11, pad=10)
    thresh = arr.max() / 2.0
    for i in range(len(labels_row)):
        for j in range(len(labels_col)):
            ax.text(j, i, str(arr[i, j]), ha='center', va='center',
                    fontsize=annot_fs, color='white' if arr[i,j] > thresh else 'black')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filepath}')

# 1. Floor confusion matrix
save_heatmap(floor_matrix, floor_labels, floor_labels,
             f'Floor Number Confusion\n{MODEL_ID}',
             f'{plots_path}/{model_name_safe}_floor_confusion.png')

# 2. Door state confusion matrix
save_heatmap(state_matrix, state_labels, state_labels,
             f'Door State (Open/Closed)\n{MODEL_ID}',
             f'{plots_path}/{model_name_safe}_door_state_confusion.png', cmap='Greens')

# 3. Type breakdown bar chart
type_data = {}
for img_type in ['complete', 'sign_only', 'wall_number']:
    sub = df[(df['category'] == 'building_sign') & (df['image_type'] == img_type)]
    fa = sub['floor_accuracy'].dropna().mean() if len(sub) > 0 else 0
    type_data[img_type] = fa

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(list(type_data.keys()), list(type_data.values()),
              color=['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336'
                     for v in type_data.values()])
ax.set_ylim(0, 1.1); ax.set_ylabel('Floor Accuracy', fontsize=11)
ax.set_title(f'Floor Accuracy by Image Type\n{MODEL_ID}', fontsize=11)
for bar, val in zip(bars, type_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
bp = f'{plots_path}/{model_name_safe}_type_breakdown.png'
plt.savefig(bp, dpi=150, bbox_inches='tight'); plt.close(); print(f'Saved: {bp}')

# 4. Room code accuracy per image
rc_df = df[df['question_id'] == 'doorroom_code'].copy()
if len(rc_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#4CAF50' if v == 1.0 else '#F44336' for v in rc_df['room_code_accuracy']]
    ax.bar(rc_df['expected_value'], rc_df['room_code_accuracy'], color=colors)
    ax.set_ylim(0, 1.2); ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(f'Room Code Accuracy per Code\n{MODEL_ID}', fontsize=11)
    ax.set_xticklabels(rc_df['expected_value'], rotation=30, ha='right')
    plt.tight_layout()
    rp = f'{plots_path}/{model_name_safe}_room_code_accuracy.png'
    plt.savefig(rp, dpi=150, bbox_inches='tight'); plt.close(); print(f'Saved: {rp}')

# 5. Overall accuracy summary bar chart
metric_names = ['Building\nAccuracy', 'Floor\nAccuracy', 'Location\nAccuracy', 'Room Code\nAccuracy', 'Detection\nEM', 'State\nEM']
metric_vals  = [
    agg_bs.get('building_accuracy'),
    agg_bs.get('floor_accuracy'),
    agg_bs.get('location_accuracy'),
    agg_dr.get('room_code_accuracy'),
    agg_dr.get('doorroom_detection_accuracy'),
    agg_dr.get('doorroom_state_accuracy')
]
valid = [(n, v) for n, v in zip(metric_names, metric_vals) if v is not None]
if valid:
    fig, ax = plt.subplots(figsize=(10, 4))
    names, vals = zip(*valid)
    colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336' for v in vals]
    bars = ax.bar(names, vals, color=colors)
    ax.set_ylim(0, 1.1); ax.set_ylabel('Score', fontsize=11)
    ax.set_title(f'All Metrics Summary\n{MODEL_ID}', fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    sp = f'{plots_path}/{model_name_safe}_accuracy_by_category.png'
    plt.savefig(sp, dpi=150, bbox_inches='tight'); plt.close(); print(f'Saved: {sp}')

print(f'\nAll results saved to {RESULTS_PATH}')
print('Download /content/results/exp2/ as ZIP when done.')